# Efficiency Simulation
_Author: Danaé Valdenaire, Philipp Schreiner_<br>
_Created: Sep. 5, 2025_<br>
_Last updated: June. 15th, 2026 by Danaé Valdenaire_

---

## Introduction

This tutorial walks you through the **efficiency simulation** using the CAIT software.
But first, a little bit of context is required.

Not all particle events end up in the final spectrum. Some fall below the **trigger
threshold**, others are removed by **quality cuts**. To correctly interpret the measured
spectrum — for example when computing dark matter exclusion limits — the probability
of an event surviving this analysis chain must be estimated as a function of energy.
This quantity is called the **survival probability**, or **efficiency**.

The efficiency is estimated by simulation: artificial pulses of known energy are
superimposed onto the real data stream at random timestamps, then processed through
the **same analysis chain** as real data. Since their true energies are known, the
surviving fraction can be extracted as a function of energy.

```{tip}
For more the algorithm, go to [**`dh.efficiency_sim_trigger_of`**](cait.mixins.SimulateMixin.efficiency_sim_trigger_of)
```

## First, you need (mock) data

In [1]:
import os
import numpy as np
import cait as ai
import cait.versatile as vai
import scipy as sp

Before starting the work, we need to create our [`DataHandler`](cait.DataHandler) that contains our data. If you are not familiar with `DataHandler`, I really advice you to have a look at the `triggering tutorial`. All the steps from the creation to the DataHandler to the triggering of the stream data are well described there. 

In [2]:
fdirh5 = "tutorial_output"
hdf5_name = "my_first_trigger"
os.makedirs(fdirh5, exist_ok=True)

record_length = 2**14

trigger_config = {
    "trigger_channels": ["Ch0"],
    "passive_channels": ["Ch1"],
    "testpulse_channels": ["TP0", "TP1"],
    "controlpulses_above": [9., 9.],
    "f_noise": 1000,
    "copy_events": True,
}
n_channels = len(trigger_config["trigger_channels"]) + len(trigger_config["passive_channels"])

stream = vai.MockStream(seed=137, rate_Hz=2)

# Set the path to the desired HDF5 file
dh = ai.DataHandler(record_length=record_length, 
                    nmbr_channels=n_channels, 
                    sample_frequency=stream.sample_frequency)

dh.set_filepath(path_h5="tutorial_output", fname="my_first_trigger", appendix=False)

DataHandler Instance created.


## Building SEV, NPS, OF

This section is detailled in the tutorial called `tutorial_sev`.

### SEV - events

In [8]:
quality_cuts = ai.cuts.LogicalCut()
quality_cuts.add_condition((dh["events/onset", 0]>-0.8)*(dh["events/onset", 0]<-0.4)); print(quality_cuts.counts())
quality_cuts.add_condition((dh["events/decay_time", 0]<40)); print(quality_cuts.counts())
quality_cuts.add_condition((dh["events/pulse_height", 0]>0.1)); print(quality_cuts.counts())

4475
4324
1025


In [9]:
dh.apply_logical_cut(cut_flag=quality_cuts.get_flag(),                                                             
                     naming='cuts_for_sev',
                     channel=0,
                     type='events',
                     delete_old=True)

Delete old cuts_for_sev dataset
Applied logical cut.


In [10]:
quality_cuts = dh["events/quality_cuts",0]
event_iterator = dh.get_event_iterator(group="events", channel=0, flag=quality_cuts)
sev = vai.SEV(event_iterator)
fitpar, _, rms = vai.apply(vai.TemplateFit(sev, bl_poly_order=1), event_iterator)
sev = vai.SEV(event_iterator[:, rms<0.005])
dh_SEV = ai.DataHandler(record_length=record_length, 
                    nmbr_channels=n_channels, 
                    sample_frequency=stream.sample_frequency)
dh_SEV.set_filepath(path_h5="tutorial_output", fname="my_SEV_OF_NPS", appendix=False)
dh_SEV.init_empty()
sev.to_dh(dh_SEV,"events","stdevent", overwrite_existing=True)

  6%|5         | 58/1025 [00:02<00:33, 28.57events/s]

DataHandler Instance created.
Successfully written stdevent with shape (1, 16384) and dtype 'float32' to group events.


### NPS

In [11]:
# performing quality cuts on noise traces

noise_cuts = ai.cuts.LogicalCut()
noise_cuts.add_condition(abs(dh["noise/pulse_height", 0])<0.05); print("Surviving after cut 1:", noise_cuts.counts())
noise_cuts.add_condition(dh["noise/variance", 0]<0.05); print("Surviving after cut 2:", noise_cuts.counts())

Surviving after cut 1: 998
Surviving after cut 2: 998


In [12]:
noise_traces = dh.get_event_iterator("noise", channel=0).with_processing(vai.RemoveBaseline())
fit_par, fit_rms = vai.apply(vai.FitBaseline(model=3, where=1.0), noise_traces)

 55%|#####5    | 554/1000 [00:02<00:01, 275.73events/s]

In [13]:
vai.ScatterPreview(fit_rms[:,0], np.arange(len(fit_rms)), noise_traces.with_processing(vai.RemoveBaseline()))

In [14]:
noise_cuts.add_condition(fit_rms[:,0] < 0.00504)

In [15]:
dh.apply_logical_cut(cut_flag=noise_cuts.get_flag(),                                   
                     naming='cuts_for_nps',
                     channel=0,
                     type='noise',
                     delete_old=True)

Delete old cuts_for_nps dataset
Applied logical cut.


In [16]:
dh_SEV = ai.DataHandler(record_length=record_length, 
                    nmbr_channels=n_channels, 
                    sample_frequency=stream.sample_frequency)
dh_SEV.set_filepath(path_h5="tutorial_output", fname="my_SEV_OF_NPS", appendix=False)
dh_SEV.init_empty()

DataHandler Instance created.


In [17]:
noise_quality_cuts = dh["noise/cuts_for_nps",0]
noise_traces = dh.get_event_iterator("noise", channel=0, flag=noise_quality_cuts)
nps = vai.NPS(noise_traces)
nps.to_dh(dh_SEV, overwrite_existing=True)

Successfully written nps with shape (1, 8193) and dtype 'float32' to group noise.


### OF

In [18]:
sev = vai.SEV().from_dh(dh_SEV, "stdevent") 
nps = vai.NPS().from_dh(dh_SEV, "nps")      
#vai.OF(sev, nps).to_dh(dh_SEV)              
of = vai.OF().from_dh(dh_SEV, "optimumfilter")  

## Template fit

This part is described in the `amplitude reconstruction tutorial`. If you don't know what is a template fit or a truncation limit, I advice you to have a look at this one first.

In [19]:
sev = vai.SEV().from_dh(dh_SEV,"events","stdevent")

In [20]:
truncation_limit = 0.3
event_iterator = dh.get_event_iterator("events", channel=0).with_processing(vai.RemoveBaseline())
fitpar, _, rms = vai.apply(vai.TemplateFit(sev=sev, bl_poly_order=1, truncation_limit=truncation_limit), event_iterator)
dh.set(group="events", sev_fit_amplitude=fitpar.T[0], overwrite_existing=True)
dh.set(group="events", sev_fit_rms=rms, overwrite_existing=True)

  1%|1         | 54/5318 [00:02<03:15, 26.92events/s]

Successfully written sev_fit_amplitude with shape (5318,) and dtype 'float32' to group events.
Successfully written sev_fit_rms with shape (5318,) and dtype 'float32' to group events.


## Energy calibration

In [26]:
tp_quality_cuts = ai.cuts.LogicalCut()
cut_rms_fit = (dh["testpulses/sev_fit_rms"]<0.0073)
cut_rms = (dh["testpulses/rms",0]<0.0053)
cut_neg_ph = (dh["testpulses/sev_fit_amplitude",0]>0)
cut_slope = np.abs(dh["testpulses/baseline_difference", 0]) < 0.01
tp_stability = dh["testpulses/testpulse_stability",0] #this flag should appear in your dh after running dh.calc_testpulse_stability

tp_quality_cuts.add_condition(cut_rms_fit*cut_neg_ph*cut_slope*cut_rms)

In [27]:
dh.apply_logical_cut(cut_flag=tp_quality_cuts.get_flag(),                                                             
                     naming='cuts_for_cleaning',
                     channel=0,
                     type='testpulses',
                     delete_old=True)

Delete old cuts_for_cleaning dataset
Applied logical cut.


In [28]:
cuts_for_cleaning = dh["testpulses/cuts_for_cleaning",0]

In [37]:
# Load the testpulse timestamps, tpa and amplitudes from the parametric fit

tp_ts = dh.get_event_iterator("testpulses", channel=0).timestamps[cuts_for_cleaning] 
tpas = dh["testpulses/testpulseamplitude",0][cuts_for_cleaning]
tp_fit_amp = dh["testpulses/sev_fit_amplitude"][cuts_for_cleaning]

In [39]:
# quick and dirty to remove outliers
unique_tp = np.unique(tp_tpa)
cond = np.ones(len(tpas), dtype=bool)
for tpa in unique_tp:
    q0, q1, q2 = np.quantile(tp_fit_amp[tpas==tpa], sp.stats.norm.cdf([-0.5,0,0.5]))
    cond[tpas==tpa] = abs((tp_fit_amp[tpas==tpa]-q1)/(q2-q0))<3
    
tp_ts = tp_ts[cond]
tpas = tpas[cond]
tp_fit_amp = tp_fit_amp[cond]

In [40]:
my_ecal = vai.EnergyCalibration(tp_x=tp_ts,
                                tp_phs=tp_fit_amp,
                                tpas=tpas,
                                testpulse_response=vai.TPRCubicSpline(kernel_length=3,remove_outliers=True),
                                transfer_function=vai.TFPchip(fix_at_yaxis=True),
                                max_x_gap=0.5)

In [41]:
event_ts = dh.get_event_iterator("events", channel=0).timestamps
event_TPE = dh["events/sev_fit_amplitude"]

TPE = my_ecal.inverse(event_ts, event_TPE)
dh.set("events", testpulse_equivalent=TPE, overwrite_existing=True) # don't forget to save your calculation in your data handler.

Successfully written testpulse_equivalent with shape (5318,) and dtype 'float32' to group events.


In [42]:
phs = my_ecal(event_ts, TPE)

In [43]:
my_ecal.to_file("my_ecal_function")

## Simulating timestamps and pulse heights

In the notebook ```new_tutorial_energy_calibration```, you learned how to perform the energy calibration of your data and save your ```e_cal``` function as a ```.json```file. If you did not completed this part, please run the ```new_tutorial_energy_calibration``` notebook first and come back here afterwards.

Let's start by loading our calibration function that enable us to convert from pulse heights to testpulse equivalent and vice versa.

In [94]:
my_ecal = vai.EnergyCalibration.from_file("my_ecal_function")

In [95]:
my_ecal.preview()

We need to generate random (sorted) timestamps. The simulation is performed on a chunk of length ```n_record_lens``` and placed according to ```record_placement```. 

```{important}
The following definition of generated timestamps is assuming default values of ```n_record_lens``` and ```record_placement```. 
```

You also need to choose the number of event you want to simulate. To get enough statistics, this number needs to be high. 

In [76]:
N_sim = 10**4

In [77]:
sim_ts = np.sort(
    sp.stats.randint.rvs(
        stream.time[0] + 6*stream.dt_us*record_length, 
        stream.time[-1] - 4*stream.dt_us*record_length, 
        size=N_sim
    )
)

In [78]:
sim_ts

array([1426321614659097, 1426321614686472, 1426321615210599, ...,
       1426325211917045, 1426325212088596, 1426325212264260],
      shape=(10000,))

The efficiency simulation can be perform in volt or in energy. In this case, we will mimic a real analysis and therefore work in energy unit. 
- First, we define the range of energy (in keV) in which we want to run the simulation. We have to stay in the linear range of the detector otherwise the pulse shape of the particle events and the simulated pulses will differ. 
- Then, we convert this energy array to testpulse equivalent using our ```e_cal``` function. The ```e_cal``` function will incorporate the instabilities of the detector over time. We start with a uniform distribution of energies but we will get a non-uniform TPE distribution. 
- The last step is to convert TPEs to pulse heights, by dividing with the CPE factor. This value is the ratio between the energy of the peak (in keV) and the position of the peak (in V) in the TPE spectrum. You should have it from the energy calibration tutorial.

## CPE factor

The $^{55}\text{Fe}$ escape peaks are respectively at 5.89 keV and 6.49 keV. We calibrate with the dominant one at 5.89 keV.

In [79]:
# Fitting the peak with a gaussian to extract the mean
min_fit, max_fit = 0.495, 0.526
fit_x = np.linspace(min_fit, max_fit, 50)

hist = vai.Histogram(TPE,bins=fit_x,
              xlabel='Testpulse equivalent (V)', 
              ylabel='Counts',
              yrange=(0,250))

gauss_fitpar = sp.stats.norm.fit(TPE[(TPE>min_fit)*(TPE<max_fit)])
hist.add_line(x=fit_x, 
              y=len(TPE[(TPE>min_fit)*(TPE<max_fit)])*np.diff(fit_x)[0]*sp.stats.norm.pdf(fit_x, *gauss_fitpar), 
              name=f"μ={gauss_fitpar[0]:.3f} \nσ={gauss_fitpar[1]:.3f} injV")

    'data': [{'showlegend': False,
              'type': 'histogram',
          …

## Trigger efficiency

Now, we are ready to run our efficiency simulation function [**`dh.efficiency_sim_trigger_of`**](cait.mixins.SimulateMixin.efficiency_sim_trigger_of).

In [102]:
energy_range = (0, 20) # Energy range to perform the simulation (keV)
energy_of_interest = energy_range[0] + np.diff(energy_range)*sp.stats.uniform.rvs(size=N_sim)
tpes_of_interest = energy_of_interest/cpe_factor
sim_phs = my_ecal(sim_ts, tpes_of_interest)

In [103]:
sim_phs[sim_phs>0]

array([0.00334065, 0.07338055, 0.00135541, ..., 0.03732443, 0.07105205,
       0.04484311], shape=(9973,))

In [104]:
dh.efficiency_sim_trigger_of(
                stream=stream,
                trigger_channels=["Ch0"],
                testpulse_channels=["TP0"],
                of=of,
                thresholds=[0.1],
                sim_ts=sim_ts[sim_phs>0],
                sim_phs=sim_phs[sim_phs>0],
                #preview=True, # Uncomment to see a preview of the trigger before you run the simulation
                sev_fitpars=sev_fitpars
            )

Triggering channel 0:   0%|          | 46/9973 [00:02<07:15, 22.79events/s]

Updated external event data dictionary.
Successfully saved event iterator reference for group trig-eff-sim.
Successfully written time_s with shape (9973,) and dtype 'int32' to group trig-eff-sim.
Successfully written time_mus with shape (9973,) and dtype 'int32' to group trig-eff-sim.
Successfully written hours with shape (9973,) and dtype 'float64' to group trig-eff-sim.
Successfully written trigger_flag with shape (1, 9973) and dtype 'bool' to group trig-eff-sim.
Successfully written flag_survived_trigger with shape (9973,) and dtype 'bool' to group trig-eff-sim.
Successfully written flag_survived_tp with shape (9973,) and dtype 'bool' to group trig-eff-sim.
Successfully written trigger_index with shape (1, 9973) and dtype 'int32' to group trig-eff-sim.
Successfully written event_timestamps with shape (9973,) and dtype 'int32' to group trig-eff-sim.
Successfully written simulated_phs with shape (1, 9973) and dtype 'float32' to group trig-eff-sim.
Successfully written reconstructed_ph

Two groups have been created in the DataHandler ```trig-eff-sim``` contains trigger information and the simulation chunks, ```events-eff-sim``` contains the particle traces which survived the procedure. Check them out using ```dh.content()```.

You can look at the stream chunks that were simulated (only contains trigger channels) $\downarrow$

In [105]:
vai.Preview(dh.get_event_iterator("trig-eff-sim").with_processing(vai.RemoveBaseline()))

You can look at the events that survived (contains trigger and passive channels) $\downarrow$

In [106]:
vai.Preview(dh.get_event_iterator("events-eff-sim").with_processing(vai.RemoveBaseline()))

Let's have a look at the survival distribution.

In [110]:
simulated_phs = dh['events-eff-sim/simulated_phs'] # pulse height array we simulated
survived_trigger = dh['events-eff-sim/flag_survived_trigger'] # events triggered by the algorithm
survived_tp = dh['events-eff-sim/flag_survived_tp'] # events that are not shadowed by a testpulse

KeyError: "Unable to synchronously open object (object 'flag_survived_trigger' doesn't exist)"

In [ ]:
vai.Histogram(
    {
        "simulated": simulated_phs, 
        "triggered": simulated_phs[survived_trigger],
        "survived tp": simulated_phs[survived_tp*survived_trigger],
    }, 
    bins=np.linspace(0, 1, 10),
    xlabel="Simulated pulse height (V)"
)

Now to get the efficiency, we need to divide the reconstructed energy by the injected energy. To recover energies from pulse heights, we need to convert with the ```e_cal``` function again. Then we binned the data and divide the histograms to get our final efficiency data.

In [ ]:
injected_voltage = simulated_phs
reconstructed_voltage = simulated_phs[survived_tp*survived_trigger]

injected_energies = my_ecal.inverse(sim_ts, injected_voltage)
reconstructed_energies = my_ecal.inverse(sim_ts, reconstructed_voltage)

In [ ]:
bins = np.linspace(0,1,200)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

In [ ]:
injected_energies_binned, bin_edges = np.histogram(injected_energies, bins=bins)

In [ ]:
reconstructed_energies_binned, bin_edges = np.histogram(reconstructed_energies, bins=bins)

In [ ]:
efficiency = reconstructed_energies_binned / np.where(injected_energies_binned !=0, injected_energies_binned, np.nan)

In [ ]:
vai.Scatter(x=bin_centers, 
            y=efficiency,
            xlabel='Pulse heights (V)',
            ylabel='Trigger efficiency',
            )

### Fitting the data with the analytical expression of the efficiency

In [ ]:
def efficiency_fct(x, p1, p2, E_thr, sigma_thr):
    return (1-p1)/2. * (1 + sp.special.erf((x - E_thr)/np.sqrt(2)/sigma_thr)) + p2

In [ ]:
popt, pcov = sp.optimize.curve_fit(efficiency_fct, 
                                   bin_centers, 
                                   efficiency,
                                   method='trf')

In [ ]:
trigger_eff_fit = efficiency_fct(bin_centers, popt[0], popt[1], popt[2], popt[3])

In [ ]:
vai.Line(x=bin_centers, 
            y=trigger_eff_fit,
            xlabel='Pulse heights (V)',
            ylabel='Trigger efficiency',
            )

## Cut efficiency

... explain how one would now apply all the quality cuts to the survived events
... give a code example of how people can plot a stacked efficiency curve for trigger and cut efficiencies

The cut efficiency is similar to the trigger effienciency. But in this case, you want to evaluate **how much particle events are removed by your quality cuts**. To do so, same method. We will use the group of simulated events ```efficiency_sim``` that we just created for the trigger efficiency and we will apply all the quality cuts we perform during the analysis.

To treat the simulated events as real events, we first need to compute the main parameters.

In [ ]:
dh.cmp("efficiency_sim")

Then, we just apply all cuts we applied during our analysis. 

In [ ]:
#TODO apply quality cuts 

decay_cuts = dh["events-eff-sim/decay_time",0]<100
delta_spike_cut = (dh["events-eff-sim/min_derivative",0]/dh["events-eff-sim/var",0])>-100
min_deriv = (dh["events-eff-sim/min_derivative",0]/dh["events-eff-sim/var",0])<-4

quality_cuts = decay_cuts*delta_spike_cut*min_deriv

If needed, we can also run the parametric fit.

In [ ]:
#TODO apply parametric fit

Then we have to follow the same steps as for the trigger efficiency:
- We take the reconstructed pulse heights and apply them all the quality cuts from our analysis.
- Then, we convert the voltages to energies with the ```e_cal``` function.
- Finally, we divide the distribution of what's comes out over what's comes in to get the efficiencies. 

```{tip}
Instead of applying all quality cuts at once, you can also add one after the other to visualise the effect of each on the final cut efficiencies. 
```

In [ ]:
reconstructed_voltage = simulated_phs[survived_tp*survived_trigger]

In [ ]:
reconstructed_voltage_after_cuts = reconstructed_voltage[quality_cuts]

In [ ]:
bins = np.linspace(0,1,200)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

In [ ]:
# Converting to energies 
reco_energies = my_ecal.inverse(sim_ts, reconstructed_voltage)
reco_energies_after_cuts = my_ecal.inverse(sim_ts, reconstructed_voltage_after_cuts)

In [ ]:
# Binning the data
reco_energies_binned = np.histogram(reco_energies, bins=bins)
reco_energies_after_cuts_binned = np.histogram(reco_energies_after_cuts, bins=bins)

In [ ]:
cut_efficiency = reco_energies_after_cuts_binned / np.where(reco_energies_binned !=0, reco_energies_binned, np.nan)

### Fitting the data with the analytical expression of the efficiency

In [ ]:
popt, pcov = sp.optimize.curve_fit(efficiency_fct, 
                                   bin_centers, 
                                   cut_efficiency,
                                   method='trf')

In [ ]:
cut_eff_fit = efficiency_fct(bin_centers, popt[0], popt[1], popt[2], popt[3])

In [ ]:
vai.Line(x=bin_centers, 
            y=cut_eff_fit,
            xlabel='Pulse heights (V)',
            ylabel='Cut efficiency',
            )

## Tips and common mistakes

... if you can think of any tips/tricks that may be useful, we can collect them here. Also mention common mistakes/errors/pitfalls.

## Other channel configurations

... maybe here we could discuss how the situation would change if we have multiple trigger channels or when using a 2d-of, e.g.
... it is probably fine to put the discussion on how to simulate SEV shifts here (and not clutter the discussion above)